# Stroke Prediction — Controlled Resampling Ratios

This notebook runs the controlled-ratio experiment from start to finish. It supports one, several, or all target ratios and evaluates all five resampling variants: SMOTE, Borderline-SMOTE, ADASYN, SMOTE-ENN, and SMOTE-Tomek. Models: Logistic Regression, XGBoost, Random Forest, and SVM. Validation: Repeated Stratified 10-Fold CV with 2 repeats; genetic algorithm (GA) tuning uses an inner 3-fold CV. Preprocessing and resampling are fitted only on each training split, and results are checkpointed after every outer fold.

## 1. Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn imbalanced-learn smote-variants xgboost scipy
!pip install -q sklearn-genetic-opt

## 2. Imports and configuration

In [ ]:
import os
import json
import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import smote_variants as sv

from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn_genetic import GASearchCV
from sklearn_genetic.space import Integer, Categorical, Continuous
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 42
OUTER_SPLITS, OUTER_REPEATS, INNER_SPLITS = 10, 2, 3
DATA_PATH = "/kaggle/input/datasets/aderahmad/stroke-dataset-raw/healthcare-dataset-stroke-data.csv"

# Evaluate all five resampling variants in the same experiment.
SAMPLERS_TO_RUN = ["SMOTE", "Borderline-SMOTE", "ADASYN", "SMOTE-ENN", "SMOTE-Tomek"]
RATIO_CONFIGS = {
    "original": None, 
    "1:1": 1.0, 
    "1.5:1": 1.5, 
    "2:1": 2.0, 
    "5:1": 5.0
}
RUN_MODE = "all"  # selected | all
SELECTED_RATIOS = ["1.5:1", "2:1"]

GA_POPULATION_SIZE, GA_GENERATIONS, GA_N_JOBS = 10, 10, -1
OUTPUT_DIR = Path("./control_ratio_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_CSV = OUTPUT_DIR / "control_ratio_fold_results.csv"
SUMMARY_CSV = OUTPUT_DIR / "control_ratio_summary.csv"
TUNING_CSV = OUTPUT_DIR / "control_ratio_best_params.csv"
RESUME_FROM_CHECKPOINT = True

if RUN_MODE not in {"selected", "all"}: raise ValueError("RUN_MODE must be selected or all.")
RATIOS_TO_RUN = list(RATIO_CONFIGS) if RUN_MODE == "all" else list(SELECTED_RATIOS)
if set(RATIOS_TO_RUN) - set(RATIO_CONFIGS): raise ValueError("One or more target ratios are not recognized.")

## 3. Models and hyperparameter search spaces

In [ ]:
RF_PARAM_GRID = {
    "model__n_estimators": Integer(50, 300),
    "model__max_depth": Integer(3, 10),
    "model__min_samples_split": Integer(2, 10),
    "model__min_samples_leaf": Integer(5, 10),
    "model__class_weight": Categorical(["balanced", "balanced_subsample", None]),
}
XGB_PARAM_GRID = {
    "model__n_estimators": Integer(50, 300), "model__max_depth": Integer(3, 10),
    "model__learning_rate": Continuous(0.01, 0.3), "model__subsample": Continuous(0.5, 1.0),
    "model__colsample_bytree": Continuous(0.5, 1.0), "model__scale_pos_weight": Continuous(1, 25),
}
LR_PARAM_GRID = {
    "model__C": Continuous(0.01, 10.0, distribution="log-uniform"),
    "model__penalty": Categorical(["l2"]), "model__solver": Categorical(["lbfgs", "saga"]),
    "model__class_weight": Categorical([None, "balanced"]),
}
SVM_PARAM_GRID = {
    "model__C": Continuous(0.01, 10.0, distribution="log-uniform"),
    "model__kernel": Categorical(["rbf"]), "model__gamma": Categorical(["scale", "auto"]),
    "model__class_weight": Categorical([None, "balanced"]),
}
BASE_MODELS = {
    "Random Forest": RandomForestClassifier(random_state=SEED, n_jobs=1),
    "XGBoost": XGBClassifier(random_state=SEED, eval_metric="logloss", n_jobs=1, tree_method="hist", verbosity=0),
    "Logistic Regression": LogisticRegression(random_state=SEED, max_iter=1000),
    "SVM": SVC(probability=True, random_state=SEED),
}
PARAM_SPACES = {
    "Random Forest": RF_PARAM_GRID, 
    "XGBoost": XGB_PARAM_GRID,
    "Logistic Regression": LR_PARAM_GRID, 
    "SVM": SVM_PARAM_GRID
}
SAMPLER_MAP = {
    "SMOTE": sv.SMOTE, 
    "Borderline-SMOTE": sv.Borderline_SMOTE1, 
    "ADASYN": sv.ADASYN,
    "SMOTE-ENN": sv.SMOTE_ENN, 
    "SMOTE-Tomek": sv.SMOTE_TomekLinks,
}
if set(SAMPLERS_TO_RUN) - set(SAMPLER_MAP):
    raise ValueError(f"Unknown sampler(s): {sorted(set(SAMPLERS_TO_RUN) - set(SAMPLER_MAP))}")
print("Target ratios:", RATIOS_TO_RUN, "| resampling variants:", SAMPLERS_TO_RUN)

## 4. Data and preprocessing

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}. Update DATA_PATH.")
raw_df = pd.read_csv(DATA_PATH)
df = raw_df.drop(columns=["id"]).copy()
y = df.pop("stroke").astype(int)
X = df.copy()
for col, mapping in {
    "ever_married": {"No": 0, "Yes": 1},
    "Residence_type": {"Rural": 0, "Urban": 1},
}.items():
    X[col] = X[col].map(mapping)

numeric_features = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi",
                    "ever_married", "Residence_type"]
categorical_features = ["gender", "work_type", "smoking_status"]
for col in numeric_features + categorical_features:
    if col not in X.columns: raise ValueError(f"Column not found: {col}")

def make_preprocessor(model_name):
    num_steps = [("imputer", SimpleImputer(strategy="mean"))]
    if model_name in {"Logistic Regression", "SVM"}:
        num_steps.append(("scaler", StandardScaler()))
    num_pipe = Pipeline(num_steps)
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", num_pipe, numeric_features),
        ("cat", cat_pipe, categorical_features),
    ], remainder="drop")

print("Shape:", X.shape, "class counts:", y.value_counts().to_dict(),
      "prevalence:", round(float(y.mean()), 5))

## 5. Controlled sampler and estimator pipeline

`smote-variants` uses the `proportion` parameter. The wrapper calculates an interpolation proportion to approximate the requested majority-to-minority ratio. Cleaning-based variants may change the final ratio; the observed ratio is recorded.

In [ ]:
class ControlledSVSampler(BaseEstimator):
    def __init__(self, sampler_name="SMOTE", target_ratio=1.0, random_state=42):
        self.sampler_name = sampler_name
        self.target_ratio = target_ratio
        self.random_state = random_state
    
    def fit_resample(self, X, y):
        from scipy import sparse

        if self.target_ratio is None:
            return X, y

        y = np.asarray(y)
        counts = Counter(y.tolist())

        if len(counts) != 2:
            raise ValueError("Ratio control supports binary classification only.")

        minority = min(counts, key=counts.get)
        majority = max(counts, key=counts.get)

        n_min = counts[minority]
        n_maj = counts[majority]

        desired_min = int(np.ceil(n_maj / float(self.target_ratio)))
        desired_min = max(n_min, desired_min)

        if desired_min <= n_min or n_maj <= n_min:
            return X, y

        proportion = float(
            np.clip(
                (desired_min - n_min) / (n_maj - n_min),
                0,
                1,
            )
        )

        sampler_cls = SAMPLER_MAP[self.sampler_name]
        sampler = sampler_cls(
            proportion=proportion,
            random_state=self.random_state,
        )

        X_input = X.toarray() if sparse.issparse(X) else np.asarray(X)

        X_res, y_res = sampler.sample(X_input, y)

        return np.asarray(X_res), np.asarray(y_res)


def make_estimator(model_name, ratio_name, sampler_name):
    steps = [("preprocess", make_preprocessor(model_name))]
    ratio = RATIO_CONFIGS[ratio_name]
    if ratio is not None:
        steps.append(("sampler", ControlledSVSampler(sampler_name, ratio, SEED)))
    steps.append(("model", BASE_MODELS[model_name]))
    return ImbPipeline(steps)

## 6. Metrics, checkpoints, and helper functions

In [ ]:
def get_positive_scores(estimator, X_test):
    probs = estimator.predict_proba(X_test)
    return probs[:, list(estimator.classes_).index(1)]

def calculate_metrics(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc_ap": average_precision_score(y_true, y_score),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

existing = pd.read_csv(CHECKPOINT_CSV) if RESUME_FROM_CHECKPOINT and CHECKPOINT_CSV.exists() else pd.DataFrame()
completed = set()
if not existing.empty:
    completed = set((str(r.sampler), str(r.ratio_name), str(r.model), int(r.repeat), int(r.fold))
                    for r in existing.itertuples())
print("Existing fold checkpoints:", len(completed))

## 7. Repeated nested cross-validation

Each outer fold is appended to a CSV checkpoint. GA optimizes the Matthews correlation coefficient (MCC) using inner stratified 3-fold CV. The current GA configuration uses `population_size=10` and `generations=10`; compare these settings with the Comparative SMOTE notebook if consistency with that experiment is required.

In [ ]:
outer_cv = RepeatedStratifiedKFold(n_splits=OUTER_SPLITS, n_repeats=OUTER_REPEATS, random_state=SEED)
t0 = time.time()
for sampler_name in SAMPLERS_TO_RUN:
    for ratio_name in RATIOS_TO_RUN:
        if ratio_name == "original" and sampler_name != SAMPLERS_TO_RUN[0]:
            continue  # Run the no-resampling baseline only once.
        sampler_label = "None" if ratio_name == "original" else sampler_name
        for model_name in BASE_MODELS:
            print(f"\n=== {ratio_name} | {model_name} | {sampler_label} ===")
            for split_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
                repeat_idx, fold_idx = split_idx // OUTER_SPLITS + 1, split_idx % OUTER_SPLITS + 1
                key = (sampler_label, ratio_name, model_name, repeat_idx, fold_idx)
                if key in completed:
                    continue
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
                estimator = make_estimator(model_name, ratio_name, sampler_name)
                inner_cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED + split_idx)
                search = GASearchCV(
                    estimator=estimator, param_grid=PARAM_SPACES[model_name],
                    population_size=GA_POPULATION_SIZE, generations=GA_GENERATIONS,
                    crossover_probability=0.8, mutation_probability=0.1, tournament_size=3,
                    elitism=True, scoring="matthews_corrcoef", cv=inner_cv,
                    n_jobs=GA_N_JOBS, verbose=False, keep_top_k=1, criteria="max",
                    algorithm="eaMuPlusLambda",
                )
                tic = time.time()
                search.fit(X_train, y_train)
                tuning_seconds = time.time() - tic
                best = search.best_estimator_
                pred, score = best.predict(X_test), get_positive_scores(best, X_test)
                metrics = calculate_metrics(y_test, pred, score)

                original_counts = Counter(y_train.tolist())
                pre_ratio = max(original_counts.values()) / min(original_counts.values())
                post_ratio = np.nan
                if ratio_name != "original":
                    # Recalculate the post-resampling training ratio for logging only; it is not used for test predictions.
                    Xt = best.named_steps["preprocess"].transform(X_train)
                    Xr, yr = best.named_steps["sampler"].fit_resample(Xt, y_train.to_numpy())
                    cc = Counter(np.asarray(yr).tolist())
                    post_ratio = max(cc.values()) / min(cc.values())

                row = {
                    "sampler": sampler_label, "ratio_name": ratio_name,
                    "target_ratio_majority_minority": RATIO_CONFIGS[ratio_name],
                    "model": model_name, "repeat": repeat_idx, "fold": fold_idx,
                    "n_train": len(train_idx), "n_test": len(test_idx),
                    "train_original_ratio": pre_ratio, "train_post_resampling_ratio": post_ratio,
                    "best_inner_mcc": float(search.best_score_), "tuning_seconds": tuning_seconds,
                    **metrics,
                }
                pd.DataFrame([row]).to_csv(
                    CHECKPOINT_CSV, mode="a", header=not CHECKPOINT_CSV.exists(), index=False
                )
                with open(TUNING_CSV, "a", encoding="utf-8") as f:
                    if not TUNING_CSV.exists() or TUNING_CSV.stat().st_size == 0:
                        f.write("sampler,ratio_name,model,repeat,fold,best_params,best_inner_mcc,tuning_seconds\n")
                    f.write(",".join([
                        sampler_label, ratio_name, model_name, str(repeat_idx), str(fold_idx),
                        '"' + json.dumps(search.best_params_, default=str).replace('"', '""') + '"',
                        str(float(search.best_score_)), str(tuning_seconds)
                    ]) + "\n")
                completed.add(key)
                print(f"rep={repeat_idx} fold={fold_idx} BA={metrics['balanced_accuracy']:.4f} "
                      f"MCC={metrics['mcc']:.4f} AP={metrics['pr_auc_ap']:.4f} "
                      f"time={tuning_seconds:.1f}s")
    print(f"Elapsed: {(time.time()-t0)/60:.1f} minutes")

## 8. Summary

In [ ]:
results = pd.read_csv(CHECKPOINT_CSV)
results = results[(results["sampler"].isin(SAMPLERS_TO_RUN + ["None"])) &
                  (results["ratio_name"].isin(RATIOS_TO_RUN))].copy()
metrics = ["balanced_accuracy", "mcc", "roc_auc", "pr_auc_ap", "precision",
           "recall_sensitivity", "specificity", "f1", "tn", "fp", "fn", "tp", "tuning_seconds"]
summary = results.groupby(["sampler", "ratio_name", "model"])[metrics].agg(["mean", "std"])
summary.columns = [f"{a}_{b}" for a, b in summary.columns]
summary = summary.reset_index()
summary.to_csv(SUMMARY_CSV, index=False)
display(summary)

expected = OUTER_SPLITS * OUTER_REPEATS
check = results.groupby(["sampler", "ratio_name", "model"]).size().rename("completed_folds").reset_index()
check["expected_folds"] = expected
check["complete"] = check["completed_folds"] == expected
display(check)

## Important notes

- `original` means no resampling. Other settings specify the target majority-to-minority ratio.
- Preprocessing is fitted inside the pipeline. The sampler is applied only to training folds, including inner CV; test folds are never resampled.
- The primary metrics consistent with the earlier article are Balanced Accuracy, MCC, and PR-AUC/AP. Additional metrics are included to diagnose performance on the stroke class.
- Because One-Hot Encoding precedes resampling, a sampler may generate fractional values in one-hot columns. This follows the current preprocessing approach but is not an ideal semantic representation of categorical data.
- For `SMOTE-ENN` and `SMOTE-Tomek`, the final ratio may differ from the target because of the cleaning stage; inspect `train_post_resampling_ratio`.
- The full experiment is computationally expensive: 1 no-resampling baseline + 4 resampling ratios × 5 variants, across 4 models and 20 outer folds, with GA and inner 3-fold CV. Test one ratio and one model with a debug configuration before running the full experiment.
- This notebook has not been tested end-to-end in Kaggle. Compatibility may depend on the installed versions of `smote-variants` and `sklearn-genetic-opt.